---

# ~ BirdCLEF 2026: Perch Embedding Probe ~
*Wildlife species identification from Pantanal soundscapes*

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Loading**](#3)
4. [**Perch Label Mapping**](#4)
5. [**Perch Inference**](#5)
6. [**Prior Fusion**](#6)
7. [**Out-of-Fold Meta-features**](#7)
8. [**Embedding Probes**](#8)
9. [**Test Inference**](#9)
10. [**Submission**](#10)

---

# Introduction <a name="1"></a>

---

**BirdCLEF 2026** is a multi-label classification task: identify 234 wildlife species from 5-second segments of passive acoustic recordings in Brazil's Pantanal wetlands. Performance is measured by macro-averaged ROC-AUC over species that have at least one positive in the test ground truth. The submission notebook must run CPU-only within 90 minutes and without internet access.

---

## Approach

Google's **Perch v2** serves as a frozen feature extractor. No fine-tuning is performed; the 59 fully-labeled soundscape files provide enough signal to train lightweight linear probes on frozen embeddings in a few minutes on CPU.

```
Perch v2 (frozen)
    |
    +-- raw logits (234 classes, mapped from Perch's 14k-class vocab)
    |        |
    |        +-- site x hour Bayesian prior fusion
    |                 |
    |                 v
    |            fused logits  (base scores)
    |
    +-- 1536-dim embeddings
             |
             v
         PCA (64 dims) + per-class LogisticRegression probes
             |
             v
         final = 0.6 * base + 0.4 * probe -> sigmoid -> submission.csv
```

---

## Reference Shoutouts

- [Perch v2 Starter: Train + Infer](https://www.kaggle.com/code/jaejohn/perch-v2-starter-train-infer) by jaejohn
- [bc26-tensorflow-2-20-0](https://www.kaggle.com/datasets/kdmitrie/bc26-tensorflow-2-20-0) by kdmitrie

---

# Initialization <a name="2"></a>

---

The runtime environment is configured by installing TensorFlow 2.20, loading all libraries, and defining a single `Settings` class that centralises every path and hyperparameter. The random seed is fixed for reproducibility.

## Environment

In [1]:
import os, platform, sys
print('Python :', sys.version)
print('OS     :', platform.system(), platform.release())
print('CWD    :', os.getcwd())

Python : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
OS     : Linux 6.6.113+
CWD    : /kaggle/working


---

## Install

In [2]:
import subprocess, sys
from pathlib import Path

# ==========================================
# TAHAP 1: INSTALASI LIBRARY (INTERNET ON)
# ==========================================
print("--- TAHAP 1: INSTALASI LIBRARY ---")

# 1. Instalasi TensorFlow 2.20 (Wajib untuk Google Perch)
_WHL = Path('/kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel')
if _WHL.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
        str(_WHL / 'tensorboard-2.20.0-py3-none-any.whl')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
        str(_WHL / 'tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl')], check=True)
    print('Installed TF 2.20.0 from Kaggle dataset wheels.')
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        'tensorflow==2.20.0', 'tensorboard==2.20.0'], check=True)
    print('Installed TF 2.20.0 from PyPI.')

# 2. Instalasi Arsitektur Vision & Audio Augmentation
# timm digunakan untuk mengakses model Swin Transformer dan DINOv2
# audiomentations digunakan untuk teknik augmentasi audio tingkat lanjut
print("Installing timm and audiomentations...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'audiomentations'], check=True)
print('Installed timm and audiomentations.')

--- TAHAP 1: INSTALASI LIBRARY ---
Installed TF 2.20.0 from Kaggle dataset wheels.
Installing timm and audiomentations...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 13.9 MB/s eta 0:00:00
Installed timm and audiomentations.


---

## Libraries

In [3]:
import gc
import json
import random
import re
import warnings
import os
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf

# PyTorch Ecosystem untuk DINOv2 Large & Swin Transformer
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T # Digunakan untuk FrequencyMasking pada spektrogram
import timm

# Audiomentations untuk augmentasi waveform (suara mentah)
# FrequencyMask dihapus karena teknik ini dilakukan di level spektrogram
from audiomentations import Compose, AddGaussianNoise, TimeMask, PitchShift, Shift

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Catatan: Hapus baris di bawah ini jika Anda ingin menggunakan GPU 
# untuk melatih model Mega-Probe atau DINOv2 Large.
os.environ['CUDA_VISIBLE_DEVICES'] = '' 

tf.experimental.numpy.experimental_enable_numpy_behavior()

print('TensorFlow :', tf.__version__)
print('PyTorch    :', torch.__version__)
print('Timm       :', timm.__version__)
print('NumPy      :', np.__version__)
print('Pandas     :', pd.__version__)

TensorFlow : 2.20.0
PyTorch    : 2.10.0+cu128
Timm       : 1.0.25
NumPy      : 2.0.2
Pandas     : 2.3.3


---

## Settings

> `Settings` holds every path and hyperparameter in one place.

> **Cache auto-detection**: the Perch inference cache is checked in candidate directories in order; the first hit wins. Compatible with V1 cache format (`full_perch_meta.parquet` + `full_perch_arrays.npz` with keys `scores_full_raw` / `emb_full`). The OOF is always recomputed from the cached embeddings using site-level folds.

In [4]:
class Settings:
    # Mode ------------------------------------------------------------------
    # 'submit' : uses frozen hyperparameters, skips grid search
    # 'train'  : runs probe parameter grid search and prints OOF diagnostics
    MODE = 'submit'
    SEED = 42
    
    # Competition paths -----------------------------------------------------
    _KAGGLE_BASE = Path('/kaggle/input/competitions/birdclef-2026')
    _LOCAL_BASE  = Path('../dataset')
    BASE = _KAGGLE_BASE if _KAGGLE_BASE.exists() else _LOCAL_BASE

    # Perch v2 SavedModel ---------------------------------------------------
    MODEL_DIR = Path(
        '/kaggle/input/models/google/bird-vocalization-classifier'
        '/tensorflow2/perch_v2_cpu/1'
    )

    # Perch cache (V1 format: full_perch_meta.parquet + full_perch_arrays.npz)
    # Keys expected: scores_full_raw, emb_full
    _CACHE_CANDIDATES = [
        Path('/kaggle/input/datasets/jaejohn/perch-meta'),  # attached Kaggle dataset
        Path('../input'),                                    # local dev
        Path('/kaggle/working/cache'),                       # freshly generated this session
    ]
    CACHE_DIR = next(
        (
            d for d in _CACHE_CANDIDATES
            if (d / 'full_perch_meta.parquet').exists()
            and (d / 'full_perch_arrays.npz').exists()
        ),
        None,
    )
    CACHE_EXISTS = CACHE_DIR is not None
    WORK_CACHE_DIR = Path('/kaggle/working/cache')

    # Audio -----------------------------------------------------------------
    SR             = 32_000
    WINDOW_SEC     = 5
    WINDOW_SAMPLES = SR * WINDOW_SEC   # 160,000 samples
    FILE_SAMPLES   = 60 * SR           # 1,920,000 samples
    N_WINDOWS      = 12                # 5-second windows per 60-second file
    BATCH_FILES    = 16
    DRYRUN_N_FILES = 20                # dry-run when hidden test is not mounted

    # Prior fusion (frozen from OOF tuning) ---------------------------------
    LAMBDA_EVENT         = 0.4   # birds: Perch logit is already informative
    LAMBDA_TEXTURE       = 1.0   # frogs/insects: presence is location-determined
    LAMBDA_PROXY_TEXTURE = 0.8   # genus proxy is noisier than a direct mapping
    SMOOTH_TEXTURE_ALPHA = 0.35  # temporal smoothing within each 60-second file

    # Embedding probe (frozen from OOF tuning) ------------------------------
    PROBE_PCA_DIM = 32
    PROBE_MIN_POS = 8
    PROBE_C       = 0.25
    PROBE_ALPHA   = 0.40


CFG = Settings()
CFG.WORK_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'MODE         : {CFG.MODE}')
print(f'BASE         : {CFG.BASE}')
print(f'CACHE_EXISTS : {CFG.CACHE_EXISTS}')
print(f'CACHE_DIR    : {CFG.CACHE_DIR}')

MODE         : submit
BASE         : /kaggle/input/competitions/birdclef-2026
CACHE_EXISTS : True
CACHE_DIR    : /kaggle/input/datasets/jaejohn/perch-meta


---

## Seed

In [5]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.SEED)

---

# Data Loading <a name="3"></a>

---

The four metadata files are loaded and the 739 exact duplicate rows are stripped from `train_soundscapes_labels.csv`. Multi-species segments are aggregated into a union label list and the multi-hot label matrix `Y_SC` is built over all soundscape rows. Only the 59 fully-labeled files, those with all 12 windows annotated, are used for OOF and probe training.

In [6]:
taxonomy        = pd.read_csv(CFG.BASE / 'taxonomy.csv')
train_meta      = pd.read_csv(CFG.BASE / 'train.csv')
soundscape_raw  = pd.read_csv(CFG.BASE / 'train_soundscapes_labels.csv')
sample_sub      = pd.read_csv(CFG.BASE / 'sample_submission.csv')

soundscape_lbls = soundscape_raw.drop_duplicates().reset_index(drop=True)

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES      = len(PRIMARY_LABELS)
label_to_idx   = {c: i for i, c in enumerate(PRIMARY_LABELS)}

print(f'taxonomy species     : {len(taxonomy)}')
print(f'train recordings     : {len(train_meta):,}')
print(f'soundscape rows      : {len(soundscape_lbls):,} unique '
      f'(dropped {len(soundscape_raw) - len(soundscape_lbls):,} duplicates)')
print(f'submission classes   : {N_CLASSES}')

taxonomy species     : 234
train recordings     : 35,549
soundscape rows      : 739 unique (dropped 739 duplicates)
submission classes   : 234


---

## Parse Labels and Build Label Matrix

In [7]:
FNAME_RE = re.compile(
    r'BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg'
)

def parse_labels(x):
    if pd.isna(x):
        return []
    return [t.strip() for t in str(x).split(';') if t.strip()]

def union_labels(series):
    return sorted(set(lbl for x in series for lbl in parse_labels(x)))

def parse_soundscape_filename(name):
    m = FNAME_RE.match(name)
    if not m:
        return {'site': None, 'hour_utc': -1}
    _, site, _, hms = m.groups()
    return {'site': site, 'hour_utc': int(hms[:2])}

# Aggregate labels per 5-second window and attach metadata
sc_clean = (
    soundscape_lbls
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
sc_clean['end_sec'] = pd.to_timedelta(sc_clean['end']).dt.total_seconds().astype(int)
sc_clean['row_id']  = (
    sc_clean['filename'].str.replace('.ogg', '', regex=False)
    + '_' + sc_clean['end_sec'].astype(str)
)
meta_cols = sc_clean['filename'].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean  = pd.concat([sc_clean, meta_cols], axis=1)

# Identify fully-labeled files (all 12 windows annotated)
wpf        = sc_clean.groupby('filename').size()
full_files = sorted(wpf[wpf == CFG.N_WINDOWS].index.tolist())
sc_clean['file_fully_labeled'] = sc_clean['filename'].isin(full_files)

# Multi-hot label matrix for all soundscape rows
Y_SC = np.zeros((len(sc_clean), N_CLASSES), dtype=np.uint8)
for i, labels in enumerate(sc_clean['label_list']):
    for lbl in labels:
        if lbl in label_to_idx:
            Y_SC[i, label_to_idx[lbl]] = 1

# Ordered truth for fully-labeled files
full_truth = (
    sc_clean[sc_clean['file_fully_labeled']]
    .sort_values(['filename', 'end_sec'])
    .reset_index(drop=False)
)
Y_FULL_TRUTH = Y_SC[full_truth['index'].to_numpy()]

print(f'Fully-labeled files : {len(full_files)}')
print(f'Trusted windows     : {len(full_truth)}')
print(f'Active classes      : {int((Y_FULL_TRUTH.sum(axis=0) > 0).sum())}')

Fully-labeled files : 59
Trusted windows     : 708
Active classes      : 71


---

# Perch Label Mapping <a name="4"></a>

---

Perch v2 was trained on 14,795 classes identified by scientific name. Each of the 234 competition species is mapped to its Perch class index by joining on `scientific_name`. Species with no direct match are left unmapped or, for unmapped amphibians, assigned a genus-level proxy from any Perch class sharing the same genus.

## Load Perch and Map Labels

In [8]:
print('Loading Perch model...')
birdclassifier = tf.saved_model.load(str(CFG.MODEL_DIR))
infer_fn       = birdclassifier.signatures['serving_default']
print('Perch loaded.')

bc_labels = (
    pd.read_csv(CFG.MODEL_DIR / 'assets' / 'labels.csv')
    .reset_index()
    .rename(columns={'index': 'bc_index', 'inat2024_fsd50k': 'scientific_name'})
)
NO_LABEL_INDEX = len(bc_labels)

taxonomy_ = taxonomy.copy()
taxonomy_['scientific_name'] = taxonomy_['scientific_name'].astype(str)
mapping = taxonomy_.merge(
    bc_labels[['scientific_name', 'bc_index']],
    on='scientific_name', how='left'
)
mapping['bc_index'] = mapping['bc_index'].fillna(NO_LABEL_INDEX).astype(int)

label_to_bc   = mapping.set_index('primary_label')['bc_index']
BC_INDICES    = np.array([int(label_to_bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)

MAPPED_MASK       = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS        = np.where(MAPPED_MASK)[0].astype(np.int32)
UNMAPPED_POS      = np.where(~MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)

print(f'Mapped   : {MAPPED_MASK.sum()} / {N_CLASSES}')
print(f'Unmapped : {(~MAPPED_MASK).sum()}')

Loading Perch model...


2026-04-27 14:56:03.637672: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


Perch loaded.
Mapped   : 203 / 234
Unmapped : 31


---

## Class Type Indices

Species are split into texture classes (Amphibia, Insecta) and event classes (everything else). The distinction drives different prior fusion lambdas: frogs and insects are location-determined recurring textures, whereas birds are sparse acoustic events.

In [9]:
CLASS_NAME_MAP = taxonomy_.set_index('primary_label')['class_name'].to_dict()
TEXTURE_TAXA   = {'Amphibia', 'Insecta'}
ACTIVE_CLASSES = [PRIMARY_LABELS[i] for i in np.where(Y_SC.sum(axis=0) > 0)[0]]

idx_active_texture = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) in TEXTURE_TAXA],
    dtype=np.int32
)
idx_active_event = np.array(
    [label_to_idx[c] for c in ACTIVE_CLASSES if CLASS_NAME_MAP.get(c) not in TEXTURE_TAXA],
    dtype=np.int32
)

idx_mapped_active_texture  = idx_active_texture[MAPPED_MASK[idx_active_texture]]
idx_mapped_active_event    = idx_active_event[MAPPED_MASK[idx_active_event]]
idx_unmapped_active_texture = idx_active_texture[~MAPPED_MASK[idx_active_texture]]
idx_unmapped_active_event   = idx_active_event[~MAPPED_MASK[idx_active_event]]
idx_unmapped_inactive = np.array(
    [i for i in UNMAPPED_POS if PRIMARY_LABELS[i] not in ACTIVE_CLASSES], dtype=np.int32
)

print(f'Active texture classes : {len(idx_active_texture)}')
print(f'Active event classes   : {len(idx_active_event)}')
print(f'Unmapped inactive      : {len(idx_unmapped_inactive)}')

Active texture classes : 42
Active event classes   : 33
Unmapped inactive      : 2


---

## Amphibian Genus Proxies

For unmapped amphibians that are not insect sonotypes, Perch's label list is searched for any class sharing the same genus. The maximum logit across genus matches serves as a proxy score, which is better than leaving those columns at zero.

In [10]:
unmapped_df = mapping[mapping['bc_index'] == NO_LABEL_INDEX].copy()
unmapped_non_sonotype = unmapped_df[
    ~unmapped_df['primary_label'].astype(str).str.contains('son', na=False)
].copy()

proxy_map = {}
for _, row in unmapped_non_sonotype.iterrows():
    genus = str(row['scientific_name']).split()[0]
    hits  = bc_labels[
        bc_labels['scientific_name'].str.match(rf'^{re.escape(genus)}\s', na=False)
    ]
    if len(hits) > 0:
        proxy_map[str(row['primary_label'])] = hits['bc_index'].astype(int).tolist()

SELECTED_PROXY_TARGETS   = sorted([t for t in proxy_map if CLASS_NAME_MAP.get(t) == 'Amphibia'])
selected_proxy_pos       = np.array([label_to_idx[c] for c in SELECTED_PROXY_TARGETS], dtype=np.int32)
selected_proxy_pos_to_bc = {
    label_to_idx[t]: np.array(proxy_map[t], dtype=np.int32) for t in SELECTED_PROXY_TARGETS
}

idx_selected_proxy_active_texture  = np.intersect1d(selected_proxy_pos, idx_active_texture)
idx_selected_prioronly_active_texture = np.setdiff1d(idx_unmapped_active_texture, selected_proxy_pos)
idx_selected_prioronly_active_event   = np.setdiff1d(idx_unmapped_active_event, selected_proxy_pos)

print(f'Frog proxy targets : {SELECTED_PROXY_TARGETS}')

Frog proxy targets : ['1491113', '1595929', '25073']


---

# Perch Inference <a name="5"></a>

---

The 59 fully-labeled training soundscapes are run through Perch to obtain raw logits and 1536-dimensional embeddings. If a compatible cache exists (V1 format: `full_perch_meta.parquet` + `full_perch_arrays.npz`), it is loaded directly and inference is skipped to save runtime.

## Audio and Inference Helpers

In [11]:
spectrogram_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=CFG.SR,
    n_fft=2048,
    hop_length=512,
    n_mels=224,
    f_min=20,
    f_max=16000
)

def get_spec_tensor(y_batch):
    with torch.no_grad():
        spec = spectrogram_transform(torch.from_numpy(y_batch))
        spec = torchaudio.transforms.AmplitudeToDB()(spec)
        spec_min, spec_max = spec.min(), spec.max()
        spec = (spec - spec_min) / (spec_max - spec_min + 1e-8)
        spec = F.interpolate(spec.unsqueeze(1), size=(224, 224), mode='bilinear')
        return spec.repeat(1, 3, 1, 1).float()

def infer_perch_batch(paths, verbose=True):
    paths = [Path(p) for p in paths]
    n_files = len(paths)
    n_rows = n_files * CFG.N_WINDOWS

    row_ids = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites = np.empty(n_rows, dtype=object)
    hours = np.empty(n_rows, dtype=np.int16)
    scores = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embeddings = np.zeros((n_rows, 3584), dtype=np.float32)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    dinov2 = torch.hub.load(str(CFG.DINOV2_PATH), 'vit_large_patch14_dinov2', 
                            source='local', pretrained=False).to(device).eval()
    
    swin = timm.create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=0).to(device).eval()

    write_row = 0
    itr = tqdm(range(0, n_files, CFG.BATCH_FILES), desc='Inference', disable=not verbose)

    for start in itr:
        batch = paths[start:start + CFG.BATCH_FILES]
        bn = len(batch)
        x_audio = np.empty((bn * CFG.N_WINDOWS, CFG.WINDOW_SAMPLES), dtype=np.float32)
        bstart = write_row

        for bi, path in enumerate(batch):
            audio = read_soundscape_60s(path)
            x_audio[bi * 12 : (bi + 1) * 12] = audio.reshape(12, CFG.WINDOW_SAMPLES)
            meta = parse_soundscape_filename(path.name)
            row_ids[write_row : write_row + 12] = [f'{path.stem}_{t}' for t in range(5, 65, 5)]
            filenames[write_row : write_row + 12] = path.name
            sites[write_row : write_row + 12] = meta['site']
            hours[write_row : write_row + 12] = meta['hour_utc']
            write_row += 12

        out_perch = infer_fn(inputs=tf.convert_to_tensor(x_audio))
        logits_p = out_perch['label'].numpy().astype(np.float32)
        emb_p = out_perch['embedding'].numpy().astype(np.float32)

        x_spec = get_spec_tensor(x_audio).to(device)
        with torch.no_grad():
            emb_d = dinov2(x_spec).cpu().numpy().astype(np.float32)
            emb_s = swin(x_spec).cpu().numpy().astype(np.float32)

        scores[bstart:write_row, MAPPED_POS] = logits_p[:, MAPPED_BC_INDICES]
        embeddings[bstart:write_row] = np.concatenate([emb_p, emb_d, emb_s], axis=1)

        for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
            scores[bstart:write_row, pos] = logits_p[:, bc_idx_arr].max(axis=1)

        del x_audio, x_spec, out_perch, logits_p, emb_p, emb_d, emb_s
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    return pd.DataFrame({'row_id': row_ids, 'filename': filenames, 'site': sites, 'hour_utc': hours}), scores, embeddings

---

## Load Cache or Run Inference

In [ ]:
# ==============================================================================
# TAHAP: MULTIMODAL INFERENCE & CACHE MANAGEMENT (VERSION 3584 - DUAL RES)
# ==============================================================================
import os, re, gc, torch, timm, torchaudio, numpy as np, pandas as pd, tensorflow as tf
import torch.nn.functional as F, soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm

# 1. Konfigurasi Settings Terpadu
class Settings:
    MODE = 'submit'
    SEED = 42
    _KAGGLE_BASE = Path('/kaggle/input/competitions/birdclef-2026')
    BASE = _KAGGLE_BASE if _KAGGLE_BASE.exists() else Path('../dataset')
    
    # Path Model Perch
    _MODEL_CANDS = [
        Path('/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8'),
        Path('/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1'),
    ]
    MODEL_DIR = next((d for d in _MODEL_CANDS if (d / 'saved_model.pb').exists() or list(d.glob('**/saved_model.pb'))), None)
    
    # Path Cache
    _CACHE_CANDS = [
        Path('/kaggle/input/datasets/jaejohn/perch-meta'), # Cache Eksternal (1536 dims)
        Path('/kaggle/working/cache'),                    # Cache Lokal (3584 dims)
    ]
    CACHE_DIR = next((d for d in _CACHE_CANDS if (d / 'full_perch_arrays.npz').exists()), None)
    CACHE_EXISTS = CACHE_DIR is not None
    WORK_CACHE_DIR = Path('/kaggle/working/cache')

    # Parameter Audio
    SR, WINDOW_SEC = 32_000, 5
    WINDOW_SAMPLES = 32_000 * 5
    FILE_SAMPLES = 60 * 32_000
    N_WINDOWS, BATCH_FILES = 12, 16

CFG = Settings()
CFG.WORK_CACHE_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Fungsi Pembantu (Helpers)
def read_soundscape_60s(path):
    y, _ = sf.read(path, dtype='float32', always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    if len(y) < CFG.FILE_SAMPLES: y = np.pad(y, (0, CFG.FILE_SAMPLES - len(y)))
    return y[:CFG.FILE_SAMPLES]

# Mel-Spectrogram Dasar (224 Mels)
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=CFG.SR, n_fft=2048, hop_length=512, n_mels=224, f_min=20, f_max=16000
).to(device)

def get_base_spec(y_batch):
    """Menghasilkan spektrogram dasar yang ternormalisasi"""
    with torch.no_grad():
        spec = mel_transform(torch.from_numpy(y_batch).to(device))
        spec = torchaudio.transforms.AmplitudeToDB().to(device)(spec)
        spec = (spec - spec.min()) / (spec.max() - spec.min() + 1e-8)
        return spec.unsqueeze(1) # [Batch, 1, Mels, Time]

# 3. Fungsi Inferensi Multimodal (Fix: Dual Resolution 518 & 224)
def infer_perch_batch(paths, verbose=True):
    print("Inisialisasi Model Multimodal (DINOv2 L @518 & Swin B @224)...")
    dinov2 = timm.create_model('vit_large_patch14_dinov2', pretrained=True, num_classes=0).to(device).eval()
    swin = timm.create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=0).to(device).eval()
    
    n_rows = len(paths) * 12
    scores = np.zeros((n_rows, N_CLASSES))
    embeddings = np.zeros((n_rows, 3584)) # Perch(1536) + DINO(1024) + Swin(1024)
    row_ids, filenames, sites, hours = [], [], [], []
    
    write_row = 0
    for start in tqdm(range(0, len(paths), CFG.BATCH_FILES), desc='Inference'):
        batch = paths[start:start + CFG.BATCH_FILES]
        x_audio = np.empty((len(batch) * 12, CFG.WINDOW_SAMPLES), dtype=np.float32)
        
        for bi, path in enumerate(batch):
            audio = read_soundscape_60s(path)
            x_audio[bi*12:(bi+1)*12] = audio.reshape(12, -1)
            meta = parse_soundscape_filename(path.name)
            row_ids.extend([f"{path.stem}_{t}" for t in range(5, 65, 5)])
            filenames.extend([path.name] * 12)
            sites.extend([meta['site']] * 12)
            hours.extend([meta['hour_utc']] * 12)
        
        # Ekstraksi Fitur Akustik (Perch - TF)
        out_p = infer_fn(inputs=tf.convert_to_tensor(x_audio))
        logits_p, emb_p = out_p['label'].numpy(), out_p['embedding'].numpy()
        
        # Ekstraksi Fitur Visual (Torch) dengan Dual Interpolation
        base_spec = get_base_spec(x_audio)
        with torch.no_grad():
            # Resolusi Tinggi (518x518) untuk DINOv2 Large
            x_dino = F.interpolate(base_spec, size=(518, 518), mode='bilinear').repeat(1, 3, 1, 1)
            emb_d = dinov2(x_dino).cpu().numpy()
            
            # Resolusi Standar (224x224) untuk Swin Transformer
            x_swin = F.interpolate(base_spec, size=(224, 224), mode='bilinear').repeat(1, 3, 1, 1)
            emb_s = swin(x_swin).cpu().numpy()
            
        scores[write_row:write_row+len(x_audio), MAPPED_POS] = logits_p[:, MAPPED_BC_INDICES]
        embeddings[write_row:write_row+len(x_audio)] = np.concatenate([emb_p, emb_d, emb_s], axis=1)
        write_row += len(x_audio)
    
    # Bersihkan Memori GPU
    del dinov2, swin, x_dino, x_swin, base_spec
    gc.collect()
    torch.cuda.empty_cache()
    
    return pd.DataFrame({'row_id': row_ids, 'filename': filenames, 'site': sites, 'hour_utc': hours}), scores, embeddings

# 4. LOGIKA UTAMA: LOAD CACHE OR RUN INFERENCE
print("--- TAHAP: LOAD CACHE OR RUN INFERENCE ---")
if CFG.CACHE_EXISTS:
    print(f'Mengecek cache di: {CFG.CACHE_DIR}')
    arr = np.load(CFG.CACHE_DIR / 'full_perch_arrays.npz')
    
    # Cek apakah cache sudah versi 3584 dimensi
    if arr['emb_full'].shape[1] == 3584:
        print('Cache Multimodal VALID (3584 dims) ditemukan. Memuat data...')
        meta_full = pd.read_parquet(CFG.CACHE_DIR / 'full_perch_meta.parquet')
        scores_full_raw = arr['scores_full_raw'].astype(np.float32)
        emb_full = arr['emb_full'].astype(np.float32)
    else:
        print(f"Update cache: {arr['emb_full'].shape[1]} -> 3584 dims...")
        full_paths = [CFG.BASE / 'train_soundscapes' / fn for fn in full_files]
        meta_full, scores_full_raw, emb_full = infer_perch_batch(full_paths)
else:
    print('Tidak ada cache. Memulai inferensi multimodal baru...')
    full_paths = [CFG.BASE / 'train_soundscapes' / fn for fn in full_files]
    meta_full, scores_full_raw, emb_full = infer_perch_batch(full_paths)

# Simpan hasil terbaru ke folder kerja (/kaggle/working/cache)
meta_full.to_parquet(CFG.WORK_CACHE_DIR / 'full_perch_meta.parquet', index=False)
np.savez_compressed(
    CFG.WORK_CACHE_DIR / 'full_perch_arrays.npz',
    scores_full_raw=scores_full_raw,
    emb_full=emb_full,
)

# 5. Finalisasi Ground Truth
full_truth_aligned = (
    full_truth.set_index('row_id')
    .loc[meta_full['row_id']]
    .reset_index(drop=False)
)
Y_FULL = Y_SC[full_truth_aligned['index'].to_numpy()]

print("-" * 30)
print(f'DONE! emb_full shape: {emb_full.shape} (Perch + DINO @518 + Swin @224)')
print(f'DONE! Y_FULL shape  : {Y_FULL.shape}')

--- TAHAP: LOAD CACHE OR RUN INFERENCE ---
Mengecek cache di: /kaggle/input/datasets/jaejohn/perch-meta
Update cache: 1536 -> 3584 dims...
Inisialisasi Model Multimodal (DINOv2 L @518 & Swin B @224)...


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Inference:   0%|          | 0/4 [00:00<?, ?it/s]

I0000 00:00:1777301787.201123     139 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


---

# Prior Fusion <a name="6"></a>

---

Raw Perch logits reflect global species probabilities from pretraining but ignore the deployment context. Bayesian prior tables are built from the soundscape metadata, covering site prevalence, hour-of-day prevalence, and their joint prevalence, each shrunk toward the global mean. The resulting logit offsets are added to the raw scores with class-type-specific weights.

## Prior Table Construction

In [ ]:
def fit_prior_tables(prior_df, Y_prior):
    prior_df = prior_df.reset_index(drop=True)
    global_p = Y_prior.mean(axis=0).astype(np.float32)

    site_keys = sorted(prior_df['site'].dropna().astype(str).unique())
    hour_keys = sorted(prior_df['hour_utc'].dropna().astype(int).unique())

    site_to_i, site_n, site_p = {}, [], []
    for s in site_keys:
        mask = prior_df['site'].astype(str).values == s
        site_to_i[s] = len(site_n)
        site_n.append(mask.sum())
        site_p.append(Y_prior[mask].mean(axis=0))
    site_n = np.array(site_n, dtype=np.float32)
    site_p = np.stack(site_p).astype(np.float32) if site_p else np.zeros((0, Y_prior.shape[1]), np.float32)

    hour_to_i, hour_n, hour_p = {}, [], []
    for h in hour_keys:
        mask = prior_df['hour_utc'].astype(int).values == h
        hour_to_i[h] = len(hour_n)
        hour_n.append(mask.sum())
        hour_p.append(Y_prior[mask].mean(axis=0))
    hour_n = np.array(hour_n, dtype=np.float32)
    hour_p = np.stack(hour_p).astype(np.float32) if hour_p else np.zeros((0, Y_prior.shape[1]), np.float32)

    sh_to_i, sh_n_list, sh_p_list = {}, [], []
    for (s, h), idx in prior_df.groupby(['site', 'hour_utc']).groups.items():
        sh_to_i[(str(s), int(h))] = len(sh_n_list)
        idx = np.array(list(idx))
        sh_n_list.append(len(idx))
        sh_p_list.append(Y_prior[idx].mean(axis=0))
    sh_n = np.array(sh_n_list, dtype=np.float32)
    sh_p = np.stack(sh_p_list).astype(np.float32) if sh_p_list else np.zeros((0, Y_prior.shape[1]), np.float32)

    return dict(
        global_p=global_p,
        site_to_i=site_to_i, site_n=site_n, site_p=site_p,
        hour_to_i=hour_to_i, hour_n=hour_n, hour_p=hour_p,
        sh_to_i=sh_to_i,     sh_n=sh_n,     sh_p=sh_p,
    )

---

## Prior Logits and Score Fusion

In [ ]:
def prior_logits(sites, hours, tables, eps=1e-4):
    n = len(sites)
    p = np.repeat(tables['global_p'][None, :], n, axis=0).astype(np.float32, copy=True)

    si  = np.fromiter((tables['site_to_i'].get(str(s), -1) for s in sites), np.int32, n)
    hi  = np.fromiter(
        (tables['hour_to_i'].get(int(h), -1) if int(h) >= 0 else -1 for h in hours),
        np.int32, n
    )
    shi = np.fromiter(
        (tables['sh_to_i'].get((str(s), int(h)), -1) if int(h) >= 0 else -1
         for s, h in zip(sites, hours)),
        np.int32, n
    )

    valid = hi >= 0
    if valid.any():
        nh = tables['hour_n'][hi[valid]][:, None]
        p[valid] = nh / (nh + 8.0) * tables['hour_p'][hi[valid]] + (1.0 - nh / (nh + 8.0)) * p[valid]

    valid = si >= 0
    if valid.any():
        ns = tables['site_n'][si[valid]][:, None]
        p[valid] = ns / (ns + 8.0) * tables['site_p'][si[valid]] + (1.0 - ns / (ns + 8.0)) * p[valid]

    valid = shi >= 0
    if valid.any():
        nsh = tables['sh_n'][shi[valid]][:, None]
        p[valid] = nsh / (nsh + 4.0) * tables['sh_p'][shi[valid]] + (1.0 - nsh / (nsh + 4.0)) * p[valid]

    np.clip(p, eps, 1.0 - eps, out=p)
    return (np.log(p) - np.log1p(-p)).astype(np.float32)


def smooth_cols(scores, cols, alpha=0.35):
    """Temporal smoothing: blend each window with its neighbours within the same file."""
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s    = scores.copy()
    view = s.reshape(-1, CFG.N_WINDOWS, s.shape[1])
    x    = view[:, :, cols]
    prev = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    nxt  = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    view[:, :, cols] = (1.0 - alpha) * x + 0.5 * alpha * (prev + nxt)
    return s


def fuse_scores(base, sites, hours, tables):
    scores = base.copy()
    prior  = prior_logits(sites, hours, tables)

    if len(idx_mapped_active_event):
        scores[:, idx_mapped_active_event] += CFG.LAMBDA_EVENT * prior[:, idx_mapped_active_event]
    if len(idx_mapped_active_texture):
        scores[:, idx_mapped_active_texture] += CFG.LAMBDA_TEXTURE * prior[:, idx_mapped_active_texture]
    if len(idx_selected_proxy_active_texture):
        scores[:, idx_selected_proxy_active_texture] += (
            CFG.LAMBDA_PROXY_TEXTURE * prior[:, idx_selected_proxy_active_texture]
        )
    if len(idx_selected_prioronly_active_event):
        scores[:, idx_selected_prioronly_active_event] = (
            CFG.LAMBDA_EVENT * prior[:, idx_selected_prioronly_active_event]
        )
    if len(idx_selected_prioronly_active_texture):
        scores[:, idx_selected_prioronly_active_texture] = (
            CFG.LAMBDA_TEXTURE * prior[:, idx_selected_prioronly_active_texture]
        )
    if len(idx_unmapped_inactive):
        scores[:, idx_unmapped_inactive] = -8.0

    scores = smooth_cols(scores, idx_active_texture, alpha=CFG.SMOOTH_TEXTURE_ALPHA)
    return scores.astype(np.float32), prior

---

# Out-of-Fold Meta-features <a name="7"></a>

---

The embedding probes take fused scores as input features, so those scores must be produced out-of-fold to avoid data leakage. The prior table for each validation fold is fitted on training folds only, excluding the entire validation site. `GroupKFold(5)` is used with groups defined by site, ensuring no site contributes to both training and validation in the same fold.

In [ ]:
def macro_auc(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    return roc_auc_score(y_true[:, keep], y_score[:, keep], average='macro')


gkf    = GroupKFold(n_splits=5)
groups = meta_full['site'].to_numpy()

oof_base  = np.zeros_like(scores_full_raw, dtype=np.float32)
oof_prior = np.zeros_like(scores_full_raw, dtype=np.float32)

for _, va_idx in tqdm(list(gkf.split(scores_full_raw, groups=groups)), desc='OOF folds'):
    va_idx    = np.sort(va_idx)
    val_sites = set(meta_full.iloc[va_idx]['site'].tolist())
    prior_m   = ~sc_clean['site'].isin(val_sites).values
    tables    = fit_prior_tables(
        sc_clean.loc[prior_m].reset_index(drop=True), Y_SC[prior_m]
    )
    oof_base[va_idx], oof_prior[va_idx] = fuse_scores(
        scores_full_raw[va_idx],
        meta_full.iloc[va_idx]['site'].to_numpy(),
        meta_full.iloc[va_idx]['hour_utc'].to_numpy(),
        tables,
    )

print(f'OOF baseline AUC (prior fusion only): {macro_auc(Y_FULL, oof_base):.6f}')

---

# Embedding Probes <a name="8"></a>

---

For each class with enough positive examples, a logistic regression probe is trained on 64-dimensional PCA embeddings augmented with sequential meta-features: the raw Perch logit, the OOF prior logit, the OOF fused score, and the previous/next/mean/max fused scores within the same 60-second file. Probes trained on these honest OOF meta-features generalise better to unseen sites than probes trained on in-fold fused scores.

## Feature Helpers

In [ ]:
def seq_features_1d(v):
    x    = v.reshape(-1, CFG.N_WINDOWS)
    prev = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    nxt  = np.concatenate([x[:, 1:], x[:, -1:]], axis=1).reshape(-1)
    return prev, nxt, np.repeat(x.mean(1), CFG.N_WINDOWS), np.repeat(x.max(1), CFG.N_WINDOWS)


def build_class_features(Z, raw_col, prior_col, base_col):
    p, n, m, mx = seq_features_1d(base_col)
    return np.concatenate(
        [Z, raw_col[:, None], prior_col[:, None], base_col[:, None],
         p[:, None], n[:, None], m[:, None], mx[:, None]],
        axis=1
    ).astype(np.float32)

---

## PCA on Embeddings

In [ ]:
emb_scaler = StandardScaler()
Z_FULL = emb_scaler.fit_transform(emb_full).astype(np.float32)

print(f'Total Dimensi Fitur Gabungan (Z_FULL) : {Z_FULL.shape[1]}')
print(f'Komposisi: Perch (1536) + DINOv2 Large (1024) + Swin (1024)')
print(f'Status: PCA dilewati untuk mempertahankan resolusi fitur multimodal.')

# Verifikasi dimensi
if Z_FULL.shape[1] != 3584:
    print(f"Peringatan: Dimensi fitur ({Z_FULL.shape[1]}) tidak sesuai dengan target multimodal (3584).")

---

## Probe Parameter Selection

In `train` mode, a small grid is swept to find the best PCA dimension, minimum positives threshold, regularisation strength, and blending weight. In `submit` mode, the frozen values from `Settings` are used directly.

In [ ]:
if CFG.MODE == 'train':
    param_grid = [
        {'pca_dim': 32, 'min_pos': 8,  'C': 0.25, 'alpha': 0.40},
        {'pca_dim': 64, 'min_pos': 8,  'C': 0.25, 'alpha': 0.40},
        {'pca_dim': 64, 'min_pos': 8,  'C': 0.25, 'alpha': 0.50},
        {'pca_dim': 64, 'min_pos': 8,  'C': 0.50, 'alpha': 0.40},
        {'pca_dim': 64, 'min_pos': 12, 'C': 0.25, 'alpha': 0.40},
        {'pca_dim': 96, 'min_pos': 8,  'C': 0.25, 'alpha': 0.40},
        {'pca_dim': 96, 'min_pos': 8,  'C': 0.50, 'alpha': 0.40},
    ]
    rows = []
    for p in tqdm(param_grid, desc='Probe grid'):
        _scaler = StandardScaler()
        _pca    = PCA(n_components=min(p['pca_dim'], emb_full.shape[0] - 1, emb_full.shape[1]))
        _Z      = _pca.fit_transform(_scaler.fit_transform(emb_full)).astype(np.float32)
        _gkf    = GroupKFold(n_splits=5)
        _groups = meta_full['site'].to_numpy()
        _oof    = oof_base.copy()
        for _, va_idx in _gkf.split(scores_full_raw, groups=_groups):
            tr_idx  = np.setdiff1d(np.arange(len(scores_full_raw)), va_idx)
            pos_cnt = Y_FULL[tr_idx].sum(axis=0)
            for ci in np.where(pos_cnt >= p['min_pos'])[0]:
                y_tr = Y_FULL[tr_idx, ci]
                if y_tr.sum() == 0 or y_tr.sum() == len(y_tr):
                    continue
                X_tr = build_class_features(
                    _Z[tr_idx], scores_full_raw[tr_idx, ci],
                    oof_prior[tr_idx, ci], oof_base[tr_idx, ci],
                )
                X_va = build_class_features(
                    _Z[va_idx], scores_full_raw[va_idx, ci],
                    oof_prior[va_idx, ci], oof_base[va_idx, ci],
                )
                clf = LogisticRegression(
                    C=p['C'], max_iter=400, solver='liblinear', class_weight='balanced'
                )
                clf.fit(X_tr, y_tr)
                pred = clf.decision_function(X_va).astype(np.float32)
                _oof[va_idx, ci] = (
                    (1.0 - p['alpha']) * oof_base[va_idx, ci] + p['alpha'] * pred
                )
        rows.append({**p, 'oof_auc': macro_auc(Y_FULL, _oof)})
    grid_df = pd.DataFrame(rows).sort_values('oof_auc', ascending=False).reset_index(drop=True)
    print(grid_df.to_string(index=False))
    best = grid_df.iloc[0]
    print(f'\nBest: pca_dim={int(best.pca_dim)} min_pos={int(best.min_pos)} '
          f'C={best.C} alpha={best.alpha}  OOF AUC={best.oof_auc:.6f}')
else:
    print(f'Using frozen probe params: pca_dim={CFG.PROBE_PCA_DIM} '
          f'min_pos={CFG.PROBE_MIN_POS} C={CFG.PROBE_C} alpha={CFG.PROBE_ALPHA}')

---

## Probe Training

In [ ]:
pos_counts  = Y_FULL.sum(axis=0)
probe_idx   = np.where(pos_counts >= CFG.PROBE_MIN_POS)[0].astype(np.int32)
probe_models = {}

for cls_idx in tqdm(probe_idx, desc='Training probes'):
    y = Y_FULL[:, cls_idx]
    if y.sum() == 0 or y.sum() == len(y):
        continue
    X = build_class_features(
        Z_FULL,
        raw_col=scores_full_raw[:, cls_idx],
        prior_col=oof_prior[:, cls_idx],
        base_col=oof_base[:, cls_idx],
    )
    clf = LogisticRegression(
        C=CFG.PROBE_C, max_iter=400, solver='liblinear', class_weight='balanced'
    )
    clf.fit(X, y)
    probe_models[cls_idx] = clf

print(f'Probe models trained : {len(probe_models)} / {N_CLASSES} classes')

---

# Pseudo-Labeling  <a name="9"></a>

---

In [ ]:
import os, re, gc, torch, timm, torchaudio, numpy as np, pandas as pd, tensorflow as tf
import torch.nn.functional as F, soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm

class Settings:
    MODE = 'submit'
    SEED = 42
    _KAGGLE_BASE = Path('/kaggle/input/competitions/birdclef-2026')
    BASE = _KAGGLE_BASE if _KAGGLE_BASE.exists() else Path('../dataset')
    _MODEL_CANDS = [
        Path('/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/8'),
        Path('/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1'),
    ]
    MODEL_DIR = next((d for d in _MODEL_CANDS if (d / 'saved_model.pb').exists() or list(d.glob('**/saved_model.pb'))), None)
    SR, WINDOW_SEC = 32_000, 5
    WINDOW_SAMPLES = 32_000 * 5
    FILE_SAMPLES = 60 * 32_000
    N_WINDOWS, BATCH_FILES = 12, 16
    DINOV2_MODEL = 'vit_large_patch14_dinov2'
    SWIN_MODEL = 'swin_base_patch4_window7_224'

CFG = Settings()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_soundscape_filename(name):
    m = FNAME_RE.match(name)
    if not m: return {"site": "Unknown", "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}

def read_soundscape_60s(path):
    y, _ = sf.read(path, dtype='float32', always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    if len(y) < CFG.FILE_SAMPLES: y = np.pad(y, (0, CFG.FILE_SAMPLES - len(y)))
    return y[:CFG.FILE_SAMPLES]

sample_sub = pd.read_csv(CFG.BASE / "sample_submission.csv")
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)

print(f"Loading Perch from: {CFG.MODEL_DIR}")
birdclassifier = tf.saved_model.load(str(CFG.MODEL_DIR))
infer_fn = birdclassifier.signatures['serving_default']

label_path = next(CFG.MODEL_DIR.glob('**/label.csv'), next(CFG.MODEL_DIR.glob('**/labels.csv')))
bc_labels = pd.read_csv(label_path).reset_index().rename(columns={"index": "bc_index"})
target_col = next((c for c in ["ebird2021", "scientific_name", "inat2024_fsd50k"] if c in bc_labels.columns), bc_labels.columns[-1])
taxonomy = pd.read_csv(CFG.BASE / 'taxonomy.csv')
mapping = taxonomy.merge(bc_labels[[target_col, "bc_index"]], left_on="primary_label", right_on=target_col, how="left")
BC_INDICES = np.array([int(mapping.set_index("primary_label")["bc_index"].fillna(len(bc_labels)).loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_POS = np.where(BC_INDICES != len(bc_labels))[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_POS].astype(np.int32)

spec_transform = torchaudio.transforms.MelSpectrogram(sample_rate=CFG.SR, n_fft=2048, hop_length=512, n_mels=224, f_min=20, f_max=16000).to(device)

def get_spec_tensor(y_batch):
    with torch.no_grad():
        spec = spec_transform(torch.from_numpy(y_batch).to(device))
        spec = torchaudio.transforms.AmplitudeToDB().to(device)(spec)
        spec = (spec - spec.min()) / (spec.max() - spec.min() + 1e-8)
        spec = F.interpolate(spec.unsqueeze(1), size=(224, 224), mode='bilinear')
        return spec.repeat(1, 3, 1, 1).float()

def infer_multimodal_batch(paths):
    print(f"Loading Backbones: {CFG.DINOV2_MODEL} & {CFG.SWIN_MODEL}")
    dinov2 = timm.create_model(CFG.DINOV2_MODEL, pretrained=True, num_classes=0).to(device).eval()
    swin = timm.create_model(CFG.SWIN_MODEL, pretrained=True, num_classes=0).to(device).eval()
    
    n_rows = len(paths) * 12
    scores, embeddings = np.zeros((n_rows, N_CLASSES)), np.zeros((n_rows, 3584))
    row_ids, filenames, sites, hours = [], [], [], []
    
    write_row = 0
    for start in tqdm(range(0, len(paths), CFG.BATCH_FILES), desc='Inference'):
        batch = paths[start:start + CFG.BATCH_FILES]
        x_audio = np.empty((len(batch) * 12, CFG.WINDOW_SAMPLES), dtype=np.float32)
        
        for bi, path in enumerate(batch):
            audio = read_soundscape_60s(path)
            x_audio[bi*12:(bi+1)*12] = audio.reshape(12, -1)
            meta = parse_soundscape_filename(path.name)
            row_ids.extend([f"{path.stem}_{t}" for t in range(5, 65, 5)])
            filenames.extend([path.name] * 12)
            sites.extend([meta['site']] * 12)
            hours.extend([meta['hour_utc']] * 12)
        
        out_p = infer_fn(inputs=tf.convert_to_tensor(x_audio))
        logits_p, emb_p = out_p['label'].numpy(), out_p['embedding'].numpy()
        x_spec = get_spec_tensor(x_audio)
        with torch.no_grad():
            emb_d = dinov2(x_spec).cpu().numpy()
            emb_s = swin(x_spec).cpu().numpy()
            
        scores[write_row:write_row+len(x_audio), MAPPED_POS] = logits_p[:, MAPPED_BC_INDICES]
        embeddings[write_row:write_row+len(x_audio)] = np.concatenate([emb_p, emb_d, emb_s], axis=1)
        write_row += len(x_audio)
        
    del dinov2, swin
    gc.collect()
    return pd.DataFrame({'row_id': row_ids, 'filename': filenames, 'site': sites, 'hour_utc': hours}), scores, embeddings

soundscape_lbls = pd.read_csv(CFG.BASE / 'train_soundscapes_labels.csv').drop_duplicates()
wpf = soundscape_lbls.groupby('filename').size()
full_files = set(wpf[wpf == 12].index.tolist())
unlabeled_paths = [p for p in sorted(list((CFG.BASE / 'train_soundscapes').glob('*.ogg'))) if p.name not in full_files]

meta_ps, scores_ps, emb_ps = infer_multimodal_batch(unlabeled_paths)

probs_ps = 1.0 / (1.0 + np.exp(-np.clip(scores_ps, -30, 30)))
Y_PSEUDO = (probs_ps > 0.90).astype(np.uint8)
active = Y_PSEUDO.sum(axis=1) > 0

df_final = meta_ps[active].reset_index(drop=True)
df_final.to_parquet("pseudo_meta.parquet")
np.savez_compressed("pseudo_features.npz", embeddings=emb_ps[active], labels=Y_PSEUDO[active])

print(f"Selesai! {Y_PSEUDO[active].shape[0]} jendela disimpan.")

---

# Test Inference <a name="9"></a>

---

With all components trained, the prior tables are re-fitted on the full labeled dataset with no fold exclusion. The complete pipeline, Perch inference, prior fusion, PCA projection, and probe blending, is then applied to the test soundscapes.

In [ ]:
final_tables = fit_prior_tables(sc_clean.reset_index(drop=True), Y_SC)

test_paths = sorted((CFG.BASE / 'test_soundscapes').glob('*.ogg'))
if len(test_paths) == 0:
    print(f'Dry-run on {CFG.DRYRUN_N_FILES} train files.')
    test_paths = sorted((CFG.BASE / 'train_soundscapes').glob('*.ogg'))[:CFG.DRYRUN_N_FILES]
else:
    print(f'Test files : {len(test_paths)}')

meta_test, scores_test_raw, emb_test = infer_perch_batch(test_paths)

test_base, test_prior = fuse_scores(
    scores_test_raw,
    meta_test['site'].to_numpy(),
    meta_test['hour_utc'].to_numpy(),
    final_tables,
)

Z_TEST = emb_scaler.transform(emb_test).astype(np.float32)

final_scores = test_base.copy()
for cls_idx, clf in tqdm(probe_models.items(), desc='Applying probes'):
    X = build_class_features(
        Z_TEST,
        raw_col=scores_test_raw[:, cls_idx],
        prior_col=test_prior[:, cls_idx],
        base_col=test_base[:, cls_idx],
    )
    pred = clf.decision_function(X).astype(np.float32)
    final_scores[:, cls_idx] = (
        (1.0 - CFG.PROBE_ALPHA) * test_base[:, cls_idx]
        + CFG.PROBE_ALPHA * pred
    )

print(f'final_scores : {final_scores.shape}')
print(f'Score range  : {final_scores.min():.3f} to {final_scores.max():.3f}')

---

# Submission <a name="10"></a>

---

Logits are converted to probabilities via sigmoid. The output shape and integrity are validated against the expected row count and column order before `submission.csv` is written.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


submission = pd.DataFrame(sigmoid(final_scores), columns=PRIMARY_LABELS)
submission.insert(0, 'row_id', meta_test['row_id'].values)
submission[PRIMARY_LABELS] = submission[PRIMARY_LABELS].astype(np.float32)

assert len(submission) == len(test_paths) * CFG.N_WINDOWS, 'Row count mismatch'
assert submission.columns.tolist() == ['row_id'] + PRIMARY_LABELS, 'Column order mismatch'
assert not submission.isna().any().any(), 'NaNs detected in submission'
assert (submission[PRIMARY_LABELS] >= 0).all().all(), 'Negative probabilities'
assert (submission[PRIMARY_LABELS] <= 1).all().all(), 'Probabilities > 1'

submission.to_csv('submission.csv', index=False)

print('submission.csv saved')
print(f'Shape : {submission.shape}')
submission.iloc[:3, :8]